In [1]:
import pandas as pd
import torch
from scipy.stats import entropy
import transformers
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM,AutoConfig
import sys
sys.path.append("./xai_scripts/tkn_coerce/")
from log_attr_scores import ProbabilisticAttributionScores
import os 
import re
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import HTML
from collections import Counter
import math
import json

import warnings
warnings.filterwarnings("ignore")

OSError: libhcoll.so.1: cannot open shared object file: No such file or directory

In [ ]:
import matplotlib.pyplot as plt
colormap_names = plt.colormaps()
print("Available Matplotlib Colormaps:")
for cmap_name in sorted(colormap_names): 
    print(cmap_name)

In [ ]:
def print_sentence_scores(sentence, scores, cmap = plt.cm.coolwarm , vmin= -1, vmax=1, legend=True, end="\n", show_plot = True):

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = cmap

    hex_colors = []
    
    colored_words = []
    for word, score in zip(sentence, scores):
        rgba = list(cmap(norm(score)))
        
        hex_color = mpl.colors.rgb2hex(rgba) # Convert RGBA to hex
        hex_color = hex_color#+"99"

        hex_colors.append(hex_color)
        
        colored_words.append(f"<span style='background-color:{hex_color}; padding:2px;'>{word}</span>")
    
    html_sentence = " ".join(colored_words)
    if show_plot:
        display(HTML(html_sentence))
        if legend:
            fig, ax = plt.subplots(figsize=(3.5, 0.1))
            cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal', alpha=0.5)
            cb.set_label("Clamped Score Range", fontsize=8)
            cb.ax.tick_params(labelsize=6)
        print(end=end)
        plt.show()
        
    else:
        return hex_colors, cmap, norm

In [ ]:
path_ = "./xai_scripts/results/one_shot_nucleus_SUB/"
path = os.path.join(path_, "torch_tensors/")
path_new_anums = "TOPP_NEW_ANUMs"

In [5]:
models_ = !ls ./xai_scripts/results/one_shot_nucleus_SUB/ALLENAI_*
models_ = Counter(["_".join(i.split("ALLENAI_")[-1].split(".csv")[0].split("_")[:-1]) for i in models_])
all_models = [k for k,v in models_.items() if v==7]
print(all_models)

model_saves_names_dict = {'gemma-3-1b-it': 'google/gemma-3-1b-it' ,
 'gemma-3-270m':"google/gemma-3-270m",
 'gpt2': "openai-community/gpt2",
 'gpt-neo-1.3B': "EleutherAI/gpt-neo-1.3B",
 'Llama-3.2-1B-Instruct': "meta-llama/Llama-3.2-1B-Instruct",
 'MobileLLM-R1-950M': "facebook/MobileLLM-R1-950M",
 'OLMo-2-0425-1B': 'allenai/OLMo-2-0425-1B',
 'OLMo-2-0425-1B-Instruct':'allenai/OLMo-2-0425-1B-Instruct',
 'Qwen2.5-1.5B-Instruct': 'Qwen/Qwen2.5-1.5B-Instruct', 
                          "Qwen3-1.7B": "Qwen/Qwen3-1.7B"}

['gemma-3-1b-it', 'gemma-3-270m', 'gpt2', 'gpt-neo-1.3B', 'Llama-3.2-1B-Instruct', 'OLMo-2-0425-1B', 'OLMo-2-0425-1B-Instruct', 'Qwen2.5-1.5B-Instruct', 'Qwen3-1.7B']


In [ ]:
all_models = [
    'gemma-3-1b-it',
 'gpt2',
 'gpt-neo-1.3B',
 'Llama-3.2-1B-Instruct',
 'OLMo-2-0425-1B',
 'OLMo-2-0425-1B-Instruct',
 'Qwen2.5-1.5B-Instruct',
    "Qwen3-1.7B"
]

In [7]:
is_nucleus = True 

In [8]:
import pickle
with open(os.path.join(path_, path_new_anums, "topp_anums_coerced_1.pkl"), "rb") as f:
    asc_dict = pickle.load(f)


asc_dict

{'gpt2': {0: tensor(-50.2491, device='cuda:0'),
  1: tensor(-37.4465, device='cuda:0'),
  2: tensor(-41.1921, device='cuda:0'),
  3: tensor(-36.9229, device='cuda:0'),
  4: tensor(-12.1985, device='cuda:0'),
  5: tensor(-41.5034, device='cuda:0'),
  6: tensor(-34.5925, device='cuda:0')},
 'gpt-neo-1.3B': {0: tensor(-5.2958, device='cuda:0'),
  1: tensor(-15.1092, device='cuda:0'),
  2: tensor(-16.7892, device='cuda:0'),
  3: tensor(-15.3707, device='cuda:0'),
  4: tensor(-15.3570, device='cuda:0'),
  5: tensor(-15.3122, device='cuda:0'),
  6: tensor(-13.2721, device='cuda:0')},
 'Llama-3.2-1B-Instruct': {0: tensor(-15.4873, device='cuda:0'),
  1: tensor(-10.4115, device='cuda:0'),
  2: tensor(-14.3466, device='cuda:0'),
  3: tensor(-14.4753, device='cuda:0'),
  4: tensor(-8.7279, device='cuda:0'),
  5: tensor(-9.8449, device='cuda:0'),
  6: tensor(-12.0521, device='cuda:0')},
 'MobileLLM-R1-950M': {0: tensor(-7.8810, device='cuda:0'),
  1: tensor(-10.8767, device='cuda:0'),
  2: tensor

In [10]:
with open('./ICML EVAL/responses_greedy_backup_march18.json', 'r') as f:
    dict_responses_old = json.load(f)

dict_responses_old

{'gemma-3-1b-it': [' "Here are some words that could fill the blank, depending on the desired tone:*',
  ' be good.The waves were beautiful, and the sun was shining brightly. He spent the morning',
  ' accomplishment.However, she quickly realized that she was lost. The dense forest was confusing, and',
  ' closely.The woman, a renowned art collector, was sitting in a plush velvet armchair, a',
  'The answer is: 1997."<end_of_turn><end_of_turn><end_of_turn><end_of_turn>',
  'a) The water bottleb) The bagc) The water bottled)',
  'Albert Einstein visited Mars in 1953.---**Explanation:**Einstein'],
 'gpt2': [" He was so excited that he didn't even think about it. He was so excited that he didn",
  ' be good.The surfers were not happy with the weather. They were worried about the',
  ' accomplishment."I\'m so glad I did it," she said. "I\'m so',
  ' closely."I\'m not sure what it is," he said. "I\'m not',
  ' What was the mission\'s mission? What was the mission\'s mission like?" Answer: "

In [11]:
dict_responses = {k:[] for k in all_models}
dict_logas = {k:[] for k in all_models}
dict_responses

{'gemma-3-1b-it': [],
 'gpt2': [],
 'gpt-neo-1.3B': [],
 'Llama-3.2-1B-Instruct': [],
 'OLMo-2-0425-1B': [],
 'OLMo-2-0425-1B-Instruct': [],
 'Qwen2.5-1.5B-Instruct': [],
 'Qwen3-1.7B': []}

In [ ]:
%%time
################
####  Plot sentence LOGAS   ###
################

response_list = []

for s_id in range(7):

    log_as_list = []

    for model_save in all_models[:]:

        path_model = os.path.join(path, model_save)

        data_folder_muilf = f"ALLENAI_MUILF_{model_save}_MU.*_SID{s_id}.pt"
        data_folder_mulf = f"ALLENAI_MULF_{model_save}_MU.*_SID{s_id}.pt"
        data_folder_asc = f"ALLENAI_{model_save}_{s_id}.csv"
        
        
        with open( os.path.join(path_, data_folder_asc), "r") as f:
            freadlines = f.readlines()
            lll = [i.replace("<|endoftext|>", " endoftext ") for idx, i in enumerate(freadlines) if idx != 0 and "LogA" not in i]
            lll = [re.sub(r'\<\|([a-z\_]+)\|\>',r"<\1>", i) for i in lll]
            lines1 = [i.replace("\n", " ") for idd, i in enumerate(lll) if i != "\n" and len(i.split("|"))>7]
            if is_nucleus: 
                linesD = [i.split("|")[5] for i in lines1[:]]
                lines = [float(asc_dict[model_save][s_id]) - float(ld) for ld in linesD]  
            else:
                if "time" in freadlines[0].split("|")[-1]:
                    lines = [float(i.split("|")[-2].split("\n")[0]) for i in lines1[:]] #logas
                else:
                    lines = [float(i.split("|")[-1].split("\n")[0]) for i in lines1[:]] #logas
                
            lines2 = [i.split("|")[4] for i in lines1[:]]
        
        
        filepaths_muilf = [i for i in os.listdir(path_model) if re.match(data_folder_muilf,i )]
        max_fp_muilf = max([int(ii.split("_")[-2].split("MU")[1]) for ii in filepaths_muilf])
        fp_muilf = [os.path.join(path_model, data_folder_muilf.replace(".*", str(i))) for i in range(max_fp_muilf + 1)]
        
        filepaths_mulf = [i for i in os.listdir(path_model) if re.match(data_folder_mulf,i )]
        max_fp_mulf = max([int(ii.split("_")[-2].split("MU")[1]) for ii in filepaths_mulf])
        fp_mulf = [os.path.join(path_model, data_folder_mulf.replace(".*", str(i))) for i in range(max_fp_mulf + 1)]
        
        
        
        ent_list = []
        ent_list_I = []
        scores = []
        for xid, x_no_resp in enumerate(fp_muilf):
            
            ent_resp = torch.load(fp_mulf[xid]) # with Response
            ent_no_resp = torch.load(x_no_resp) # does not contain Response 
        
            ent_num = torch.nn.functional.softmax( ent_resp - torch.max(ent_resp), dim = 0  ) 
            ent_denom = torch.nn.functional.softmax( ent_no_resp - torch.max(ent_no_resp), dim = 0) 
            
            ent_list.append(ent_num)
            ent_list_I.append(ent_denom) #without
                
        if "<" in lines2[0] and ">" in lines2[0]:
            lines2 = lines2[1:]
            ent_list_I = ent_list_I[1:]
            ent_list = ent_list[1:]
            lines = lines[1:]
            
        
        pk_I = torch.stack(ent_list_I, dim = 0)
        pk = torch.stack(ent_list, dim = 0)
        H_I_ = entropy(pk_I.float().numpy(), axis=-1)
        H_I = H_I_ #np.nan_to_num(H_I_, nan=0.0)
        H_ = entropy(pk.float().numpy(), axis=-1)
        H = H_ #np.nan_to_num(H_, nan=0.0)
        
        asc = np.array(lines)

        assert len(asc) == len(lines2)
    
        assert(H.shape[0] == pk.shape[0])

        
        prompt_main = lll[1].split("|")[1]
        response_main = lll[1].split("|")[2]
        response_list.append(response_main)
        
        max_y = 0.4
        min_y = -0.4

        log_as_list.append(asc)
        print_sentence_scores(lines2, asc, vmin=min_y,vmax=max_y, legend=False, end="")
        resp_print = response_main.replace("NEWLINE", "").replace("endoftext", "<|endoftext|>")
        print(f"{model_save.upper()} Response: "+resp_print, end="\n\n")
        dict_responses[model_save].append(resp_print)
        dict_logas[model_save].append(lines)
   
    print("------------------------------------------------", end="\n\n")
        

%%%%% LogA



GEMMA-3-1B-IT Response:  "Here are some words that could fill the blank, depending on the desired tone:*

%%%%% LogA



GPT2 Response:  He was so excited that he didn't even think about it. He was so excited that he didn

%%%%% LogA



GPT-NEO-1.3B Response:  He was nervous about his talk. He had been preparing and practicing for weeks, but he still wasn

%%%%% LogA



LLAMA-3.2-1B-INSTRUCT Response: "The best answer is "calm".<eot_id><start_header_id>assistant apphabetization<eot_id><start_header_id>assistant app

%%%%% LogA



...

LLAMA-3.2-1B-INSTRUCT Response:  (I know it's not possible for humans to visit Mars, but I'm curious)I'm

%%%%% LogA



OLMO-2-0425-1B Response:  <|endoftext|> Question: What is the solution?Solve 0 = -2*b - 2*b

%%%%% LogA



OLMO-2-0425-1B-INSTRUCT Response:  Answer: 1953 <|endoftext|> • Albert Einstein visited Mars in 1953. <|endoftext|> • He

%%%%% LogA



QWEN2.5-1.5B-INSTRUCT Response:  Albert Einstein never visited Mars. He was a theoretical physicist who lived from 1879 to

%%%%% time



QWEN3-1.7B Response:  - ?The question is a bit of a trick, but I need to think carefully. Albert Einstein

------------------------------------------------

CPU times: user 14min 55s, sys: 5.34 s, total: 15min
Wall time: 15.4 s
